# Exercícios de Análise de Dados — Aulas 03 e 04

**Disciplina:** Soluções em Energias Renováveis e Sustentáveis  
**Dataset:** Appliances Energy Prediction — UCI Machine Learning Repository  
**Link:** https://archive.ics.uci.edu/dataset/374/appliances+energy+prediction  

**Situação:** Uma empresa de eficiência energética analisa o comportamento de uma residência de baixo consumo. O objetivo é identificar períodos de consumo elevado dos eletrodomésticos e observar as condições de temperatura e umidade nesses momentos.

> **Instruções:**
> 1. Baixe o arquivo `energydata_complete.csv` no link acima.
> 2. No Orange Data Mining, siga a Etapa A para preparar e exportar a amostra.
> 3. Coloque o arquivo exportado (`amostra.csv`) na mesma pasta que este notebook.


## Etapa A — Orange Data Mining

1. Abra `energydata_complete.csv` no widget **File**.
2. Inspecione no **Data Table** — identifique atributos de consumo, temperatura e umidade.
3. Em **Select Columns**: mantenha `Appliances`, `lights`, pelo menos 3 de temperatura (T1, T2, T3) e 3 de umidade (RH_1, RH_2, RH_3).
4. Verifique valores ausentes no **Data Info**.
5. Use **Data Sampler** para 10% aleatório.
6. Exporte com **Save Data** como `amostra.csv`.


## Etapa B — Python / Pandas

### 1. Carregar a amostra

In [ ]:
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Carregue o arquivo exportado pelo Orange
df = pd.read_csv('amostra.csv')

print(df.head())
print('Shape:', df.shape)
df.info()
print(df.describe())


### 2. Renomear atributos

In [ ]:
df = df.rename(columns={
    'Appliances': 'Consumo_Eletrodomesticos',
    'T1': 'Temp_Sala',
    'T2': 'Temp_Cozinha',
    'T3': 'Temp_Quarto',
    'RH_1': 'Umidade_Sala',
    'RH_2': 'Umidade_Cozinha',
    'RH_3': 'Umidade_Quarto'
})

print(df.columns.tolist())


### 3. Maior consumo de eletrodomésticos

In [ ]:
max_consumo = df['Consumo_Eletrodomesticos'].max()
print(f'Maior consumo: {max_consumo} Wh')


### 4. Limiar de 70% do máximo — Alta demanda

In [ ]:
limiar = 0.70 * max_consumo
print(f'Limiar (70% do max): {limiar:.2f} Wh')

alta = df[df['Consumo_Eletrodomesticos'] > limiar]

qtd = len(alta)
pct = (qtd / len(df)) * 100
print(f'Registros acima do limiar: {qtd}')
print(f'Percentual: {pct:.2f}%')

print(alta.head())


### 5. Segundo critério: consumo elevado E temperatura acima da média

In [ ]:
temp_media = df['Temp_Sala'].mean()
print(f'Temperatura média (T1): {temp_media:.2f} °C')

alta_e_quente = df[
    (df['Consumo_Eletrodomesticos'] > limiar) &
    (df['Temp_Sala'] > temp_media)
]

qtd2 = len(alta_e_quente)
pct2 = (qtd2 / len(df)) * 100
print(f'Registros (alto consumo + alta temp): {qtd2}')
print(f'Percentual: {pct2:.2f}%')

print(alta_e_quente.head())


### 6. Comparação dos dois DataFrames

In [ ]:
print('--- Comparação ---')
print(f'Apenas alto consumo:                {qtd} registros ({pct:.2f}%)')
print(f'Alto consumo + temperatura alta:    {qtd2} registros ({pct2:.2f}%)')
print()
reducao = qtd - qtd2
print(f'Redução ao adicionar temperatura: {reducao} registros')
print('A temperatura alta restringe mais o conjunto, identificando momentos mais críticos.')


### 7. Gráfico — Distribuição do consumo

In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(df['Consumo_Eletrodomesticos'], bins=30, color='steelblue', edgecolor='white')
plt.axvline(limiar, color='red', linestyle='--', label=f'Limiar 70% ({limiar:.0f} Wh)')
plt.title('Distribuição do Consumo de Eletrodomésticos')
plt.xlabel('Consumo (Wh)')
plt.ylabel('Frequência')
plt.legend()
plt.tight_layout()
plt.savefig('grafico_exercicio.png', dpi=100)
plt.close()
print('Gráfico salvo como grafico_exercicio.png')


### Interpretação

- O consumo de eletrodomésticos se concentra em valores baixos, com poucas ocorrências de alta demanda.
- Ao combinar consumo elevado **e** temperatura acima da média, reduzimos ainda mais o conjunto — esses são os momentos mais críticos para o planejamento energético.
- A inclusão da variável de temperatura como segundo critério torna o filtro mais seletivo, focando nos cenários de maior interesse para eficiência energética.
